In [ ]:
%gui qt
%load_ext autoreload
%autoreload 2

import hmt_v3 as hmt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Preprocessing and Formatting

In [ ]:
plt.rcParams.update({
    # --- text ---
    "font.size": 14,
    # "axes.titlesize": 16,
    # "axes.labelsize": 14,
    # "xtick.labelsize": 12,
    # "ytick.labelsize": 12,
    # "legend.fontsize": 14,
    # "figure.titlesize": 20,

    # --- lines & markers ---
    "lines.linewidth": 2.5,
    "lines.markersize": 8,
    "patch.linewidth": 1.5,      # bar/patch outlines
    "axes.linewidth": 1.5,       # axis spines

    # --- ticks ---
    "xtick.major.width": 1.5,
    "ytick.major.width": 1.5,
    "xtick.major.size": 6,
    "ytick.major.size": 6,

    # --- export quality ---
    "savefig.dpi": 300,
    "figure.dpi": 100,           # on-screen only; doesn't affect saved size
    "savefig.bbox": "tight",
})

In [ ]:
me3_raw_df = pd.read_csv("test_data/k27_k27_thaw009_me3.csv")
ac_raw_df = pd.read_csv("test_data/k27_k27_thaw009_ac.csv")

me3_filtered_df = hmt.preprocess.filter_axial(me3_raw_df)
ac_filtered_df = hmt.preprocess.filter_axial(ac_raw_df)

binary_mask, me3_df, ac_df = hmt.preprocess.binarize_nucleus(me3_filtered_df, ac_filtered_df, thresh=2, bin_size=50, show_plots=True)
distance_map, contour_bands = hmt.preprocess.create_radial_contours(binary_mask, show_plots=True)

## Figure 2

In [ ]:
# RDF Plotting across nucleus
def band_of(df, contour_bands, x_min, y_min, bin_size=50):
    coords = df[["x [nm]", "y [nm]"]].to_numpy()
    x_idx = np.clip(((coords[:, 0] - x_min) / bin_size).astype(int), 0, contour_bands.shape[0] - 1)
    y_idx = np.clip(((coords[:, 1] - y_min) / bin_size).astype(int), 0, contour_bands.shape[1] - 1)
    return contour_bands[x_idx, y_idx]

x_min, y_min = hmt.preprocess.mask_origin(me3_filtered_df, ac_filtered_df)
me3_band = band_of(me3_df, contour_bands, x_min, y_min)
ac_band  = band_of(ac_df,  contour_bands, x_min, y_min)

step = 10
regions = {
    "4 Innermost Contours":   (0, 19),
    "Entire Nucleus":         (0, 99),
    "4 Outermost Contours":   (80, 99),
}

for label, (lo, hi) in regions.items():
    me3_sub = me3_df[(me3_band >= lo) & (me3_band <= hi)]
    ac_sub  = ac_df[(ac_band  >= lo) & (ac_band  <= hi)]

    me3_rdf_sub, me3_adf_sub = hmt.simulate.extract_empirical_parameters(me3_sub, sdis=500, step=step)
    ac_rdf_sub,  ac_adf_sub  = hmt.simulate.extract_empirical_parameters(ac_sub,  sdis=500, step=step)

    print(f"--- {label} ---  me3: {len(me3_sub):,} locs | ac: {len(ac_sub):,} locs")
    hmt.visualize.plot_rdf_adf(
        me3_rdf_sub, me3_adf_sub, ac_rdf_sub, ac_adf_sub, step=step,
        rdf_title=f"RDF: {label}",
        adf_title=f"ADF: {label}",
    )


## Figure 3

In [ ]:
x_min, y_min = hmt.preprocess.mask_origin(me3_filtered_df, ac_filtered_df)
me3_domains, me3_full, me3_info = hmt.simulate.generate_nucleus(
    me3_df, contour_bands, x_min, y_min, match_spacing=True, rng=np.random.default_rng(0), domain_scale=200)
ac_domains, ac_full, ac_info = hmt.simulate.generate_nucleus(
    ac_df, contour_bands, x_min, y_min, match_spacing=True, rng=np.random.default_rng(0), domain_scale=250)

In [ ]:
hmt.visualize.plot_pair_correlation(me3_info["pcf_real"], me3_info["pcf_noisy"], r_domain=me3_info["r_domain_nm"], color="green")
hmt.visualize.plot_pair_correlation(ac_info["pcf_real"], ac_info["pcf_noisy"], r_domain=ac_info["r_domain_nm"], color="red")

In [ ]:
hmt.simulate.export_to_thunderstorm(me3_domains, me3_df, "simulated_data/me3_domains.csv")
hmt.simulate.export_to_thunderstorm(me3_full, me3_df, "simulated_data/me3_full.csv")

hmt.simulate.export_to_thunderstorm(ac_domains, ac_df, "simulated_data/ac_domains.csv")
hmt.simulate.export_to_thunderstorm(ac_full, ac_df, "simulated_data/ac_full.csv")

## Figure 4

In [ ]:
crop_xmin, crop_xmax = 12000, 17000
crop_ymin, crop_ymax = 17000, 26000

me3_crop = me3_full[me3_full['x [nm]'].between(crop_xmin, crop_xmax) &
                    me3_full['y [nm]'].between(crop_ymin, crop_ymax)].copy()
ac_crop = ac_full[ac_full['x [nm]'].between(crop_xmin, crop_xmax) &
                  ac_full['y [nm]'].between(crop_ymin, crop_ymax)].copy()

calibration_area_nm2 = (crop_xmax - crop_xmin) * (crop_ymax - crop_ymin)

fraction_dfs, efficiency_fractions = hmt.preprocess.sample_lower_densities(
    me3_crop, ac_crop, num_samples=10, show_plots=False)
me3_fraction_dfs = [data[0] for data in fraction_dfs]
ac_fraction_dfs  = [data[1] for data in fraction_dfs]

In [ ]:
# Diagnostic: are true domains in this crop even geometrically separable?
# domain_scale=100 matches the domain_scale passed to generate_nucleus above.
# Run on me3_crop/ac_crop directly (density_fraction = 1.0, i.e. fraction 1/10
# in the calibration below) before spending time on the DBSCAN search.
me3_packing = hmt.simulate.diagnose_domain_packing(me3_crop, domain_scale=100, use_z=True)
ac_packing = hmt.simulate.diagnose_domain_packing(ac_crop, domain_scale=100, use_z=True)

print(f"H3K27me3: {me3_packing['n_domains']} domains, "
      f"{me3_packing['frac_overlapping']:.0%} overlap their nearest neighbour, "
      f"median NN spacing {np.median(me3_packing['nn_distance']):.0f} nm")
print(f"H3K27ac:  {ac_packing['n_domains']} domains, "
      f"{ac_packing['frac_overlapping']:.0%} overlap their nearest neighbour, "
      f"median NN spacing {np.median(ac_packing['nn_distance']):.0f} nm")

hmt.visualize.plot_domain_packing(me3_packing)
hmt.visualize.plot_domain_packing(ac_packing)

In [ ]:
print("Calibrating H3K27me3 (size-matching)...")
me3_size_result = hmt.simulate.optimize_dbscan_size_matching(
    me3_fraction_dfs, calibration_area_nm2, min_samples_range=range(5, 75, 5), show_plot=True, use_z=True, verbose=False)

print("\nCalibrating H3K27ac (size-matching)...")
ac_size_result = hmt.simulate.optimize_dbscan_size_matching(
    ac_fraction_dfs, calibration_area_nm2, min_samples_range=range(5, 75, 5), show_plot=True, use_z=True, verbose=False)

In [ ]:
hmt.visualize.plot_dbscan_calibration(
    me3_size_result, ac_size_result,
    title='DBSCAN Calibration: H3K27me3 vs H3K27ac')

In [ ]:
# Export a few density fractions of the full simulated nucleus for visual inspection in ThunderSTORM
target_fractions = [0.9, 0.6, 0.3]

for i, target in enumerate(target_fractions):
    pct = round(target * 100)
    me3_samp = me3_full.sample(frac=target, random_state=42 + i)
    ac_samp = ac_full.sample(frac=target, random_state=142 + i)

    hmt.simulate.export_to_thunderstorm(
        me3_samp, me3_df, f"simulated_data/me3_density_{pct}.csv")
    hmt.simulate.export_to_thunderstorm(
        ac_samp, ac_df, f"simulated_data/ac_density_{pct}.csv")

    print(f"Exported density fraction {target:.0%} -> "
          f"simulated_data/{{me3,ac}}_density_{pct}.csv")